# Scenario sweep

Sweep `multiplier`. Plot per-party national seat counts.

In [ ]:
import os
from pathlib import Path

def _find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("could not find repo root (no pyproject.toml in cwd or any parent)")

os.chdir(_find_repo_root())

In [ ]:
from pathlib import Path
from prediction_engine.runner import run_prediction
from prediction_engine.analysis.sweep import collect_sweep
from schema.prediction import ReformThreatConfig

pred_dir = Path("data/predictions")
snap_path = sorted(Path("data/snapshots").glob("*.sqlite"))[-1]
sweep_paths = []
for m in (0.5, 0.75, 1.0, 1.25, 1.5):
    out = run_prediction(
        snapshot_path=snap_path,
        strategy_name="reform_threat_consolidation",
        scenario=ReformThreatConfig(multiplier=m),
        out_dir=pred_dir,
        label=f"sweep_m{m:.2f}".replace(".", "p"),
    )
    sweep_paths.append(out)

summary = collect_sweep(sweep_paths)
summary

In [ ]:
import matplotlib.pyplot as plt
pivot = summary.pivot(index="multiplier", columns="party", values="seats").fillna(0)
pivot.plot(figsize=(10, 5), marker="o")
plt.ylabel("Seats")
plt.title("National seat count vs reform-threat multiplier")
plt.show()

Reform's line should be monotonically decreasing; the consolidator parties' lines monotonically increasing.